In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/phyothaw/03-nvda-news-collection-ipynb/gdelt_nvda_2020_2025.csv
/kaggle/input/notebooks/phyothaw/03-nvda-news-collection-ipynb/__results__.html
/kaggle/input/notebooks/phyothaw/03-nvda-news-collection-ipynb/__notebook__.ipynb
/kaggle/input/notebooks/phyothaw/03-nvda-news-collection-ipynb/__output__.json
/kaggle/input/notebooks/phyothaw/03-nvda-news-collection-ipynb/custom.css
/kaggle/input/notebooks/phyothaw/03-nvda-news-collection-ipynb/.virtual_documents/__notebook_source__.ipynb
/kaggle/input/notebooks/phyothaw/03-cvx-gdelt-tones-ipynb/__results__.html
/kaggle/input/notebooks/phyothaw/03-cvx-gdelt-tones-ipynb/__notebook__.ipynb
/kaggle/input/notebooks/phyothaw/03-cvx-gdelt-tones-ipynb/__output__.json
/kaggle/input/notebooks/phyothaw/03-cvx-gdelt-tones-ipynb/gdelt_cvx_2020_2025.csv
/kaggle/input/notebooks/phyothaw/03-cvx-gdelt-tones-ipynb/custom.css
/kaggle/input/notebooks/phyothaw/03-cvx-gdelt-tones-ipynb/.virtual_documents/__notebook_source__.ipynb
/kaggle/i

In [2]:
import os
import pandas as pd
import numpy as np

In [3]:
tone_files = {
    "NVDA": "/kaggle/input/notebooks/phyothaw/03-nvda-news-collection-ipynb/gdelt_nvda_2020_2025.csv",
    "GE": "/kaggle/input/notebooks/phyothaw/03-ge-gdelt-tones-ipynb/gdelt_ge_2020_2025.csv",
    "BRK-B": "/kaggle/input/notebooks/phyothaw/03-brk-b-gdelt-tone-ipynb/gdelt_brk-b_2020_2025.csv",
    "CVX": "/kaggle/input/notebooks/phyothaw/03-cvx-gdelt-tones-ipynb/gdelt_cvx_2020_2025.csv",
    "MSFT": "/kaggle/input/notebooks/phyothaw/03-msft-gdelt-tone-ipynb/gdelt_msft_2020_2025.csv"
}

In [4]:
tone_dfs = []

for expected_ticker, file_path in tone_files.items():

    print(f"Loading {expected_ticker}...")
    print(file_path)

    df = pd.read_csv(file_path)

    # Standardise column names
    df.columns = (df.columns.str.strip().str.lower())

    # Rename columns
    df = df.rename(columns={"date_only": "date", "tone": "gdelt_company_tone"})

    # Check required columns
    required_columns = {"date", "gdelt_company_tone"}
    missing_columns = required_columns.difference(df.columns)

    if missing_columns:
        raise ValueError(
            f"{expected_ticker} is missing columns: "
            f"{missing_columns}")

    # Force the expected ticker name
    df["ticker"] = expected_ticker

    # Convert date and tone
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

    df["gdelt_company_tone"] = pd.to_numeric(df["gdelt_company_tone"], errors="coerce")

    # Keep only required columns
    df = df[
        [
            "date",
            "ticker",
            "gdelt_company_tone"
        ]].copy()

    tone_dfs.append(df)

    print(f"{expected_ticker}: {len(df)} rows loaded\n")

Loading NVDA...
/kaggle/input/notebooks/phyothaw/03-nvda-news-collection-ipynb/gdelt_nvda_2020_2025.csv
NVDA: 2173 rows loaded

Loading GE...
/kaggle/input/notebooks/phyothaw/03-ge-gdelt-tones-ipynb/gdelt_ge_2020_2025.csv
GE: 2173 rows loaded

Loading BRK-B...
/kaggle/input/notebooks/phyothaw/03-brk-b-gdelt-tone-ipynb/gdelt_brk-b_2020_2025.csv
BRK-B: 2173 rows loaded

Loading CVX...
/kaggle/input/notebooks/phyothaw/03-cvx-gdelt-tones-ipynb/gdelt_cvx_2020_2025.csv
CVX: 2173 rows loaded

Loading MSFT...
/kaggle/input/notebooks/phyothaw/03-msft-gdelt-tone-ipynb/gdelt_msft_2020_2025.csv
MSFT: 2173 rows loaded



In [5]:
gdelt_combined = pd.concat(tone_dfs,ignore_index=True)

gdelt_combined = (gdelt_combined.sort_values(["ticker", "date"]).reset_index(drop=True))

print("Combined shape:", gdelt_combined.shape)

display(gdelt_combined.head())
display(gdelt_combined.tail())

Combined shape: (10865, 3)


,date,ticker,gdelt_company_tone
0,2020-01-01,BRK-B,0.4557
1,2020-01-02,BRK-B,-0.4681
2,2020-01-03,BRK-B,0.6690
3,2020-01-04,BRK-B,0.8500
4,2020-01-05,BRK-B,0.8177


,date,ticker,gdelt_company_tone
10860,2025-12-27,NVDA,1.0763
10861,2025-12-28,NVDA,0.8075
10862,2025-12-29,NVDA,0.5789
10863,2025-12-30,NVDA,0.2364
10864,2025-12-31,NVDA,-0.2209


In [6]:
print("Columns:")
print(gdelt_combined.columns.tolist())

print("\nTickers:")
print(gdelt_combined["ticker"].unique())

print("\nData types:")
print(gdelt_combined.dtypes)

print("\nMissing values:")
print(gdelt_combined.isna().sum())

Columns:
['date', 'ticker', 'gdelt_company_tone']

Tickers:
['BRK-B' 'CVX' 'GE' 'MSFT' 'NVDA']

Data types:
date                  datetime64[ns]
ticker                        object
gdelt_company_tone           float64
dtype: object

Missing values:
date                  0
ticker                0
gdelt_company_tone    0
dtype: int64


In [7]:
invalid_dates = gdelt_combined[gdelt_combined["date"].isna()]

print("Invalid date rows:", len(invalid_dates))
display(invalid_dates.head())

Invalid date rows: 0


,date,ticker,gdelt_company_tone


In [8]:
duplicate_mask = gdelt_combined.duplicated(
    subset=["ticker", "date"],
    keep=False)

duplicate_rows = gdelt_combined[duplicate_mask].sort_values(["ticker", "date"])

print("Duplicate ticker-date rows:",len(duplicate_rows))
display(duplicate_rows.head(20))

Duplicate ticker-date rows: 0


,date,ticker,gdelt_company_tone


In [9]:
coverage_summary = (
    gdelt_combined
    .groupby("ticker")
    .agg(
        first_date=("date", "min"),
        last_date=("date", "max"),
        total_rows=("date", "size"),
        missing_tone=(
            "gdelt_company_tone",
            lambda x: x.isna().sum())).reset_index())

display(coverage_summary)

,ticker,first_date,last_date,total_rows,missing_tone
0,BRK-B,2020-01-01,2025-12-31,2173,0
1,CVX,2020-01-01,2025-12-31,2173,0
2,GE,2020-01-01,2025-12-31,2173,0
3,MSFT,2020-01-01,2025-12-31,2173,0
4,NVDA,2020-01-01,2025-12-31,2173,0


In [10]:
gdelt_combined["year"] = (gdelt_combined["date"].dt.year)

yearly_counts = (
    gdelt_combined
    .groupby(["ticker", "year"])
    .size()
    .reset_index(name="row_count"))

yearly_pivot = yearly_counts.pivot(
    index="year",
    columns="ticker",
    values="row_count"
)

display(yearly_pivot)

ticker,BRK-B,CVX,GE,MSFT,NVDA
year,,,,,
2020,365,365,365,365,365
2021,365,365,365,365,365
2022,365,365,365,365,365
2023,364,364,364,364,364
2024,366,366,366,366,366
2025,348,348,348,348,348


In [11]:
calendar_dates = pd.date_range(
    start="2020-01-01",
    end="2025-12-31",
    freq="D")

missing_dates_by_ticker = {}

for ticker in sorted(gdelt_combined["ticker"].unique()):

    ticker_dates = set(
        gdelt_combined.loc[
            gdelt_combined["ticker"] == ticker,
            "date"])

    missing_dates = [
        date for date in calendar_dates
        if date not in ticker_dates]

    missing_dates_by_ticker[ticker] = missing_dates

    print(f"\n{ticker}: {len(missing_dates)} missing dates")
    print(missing_dates)


BRK-B: 19 missing dates
[Timestamp('2020-10-20 00:00:00'), Timestamp('2023-03-23 00:00:00'), Timestamp('2025-06-15 00:00:00'), Timestamp('2025-06-16 00:00:00'), Timestamp('2025-06-17 00:00:00'), Timestamp('2025-06-18 00:00:00'), Timestamp('2025-06-19 00:00:00'), Timestamp('2025-06-20 00:00:00'), Timestamp('2025-06-21 00:00:00'), Timestamp('2025-06-22 00:00:00'), Timestamp('2025-06-23 00:00:00'), Timestamp('2025-06-24 00:00:00'), Timestamp('2025-06-25 00:00:00'), Timestamp('2025-06-26 00:00:00'), Timestamp('2025-06-27 00:00:00'), Timestamp('2025-06-28 00:00:00'), Timestamp('2025-06-29 00:00:00'), Timestamp('2025-06-30 00:00:00'), Timestamp('2025-07-01 00:00:00')]

CVX: 19 missing dates
[Timestamp('2020-10-20 00:00:00'), Timestamp('2023-03-23 00:00:00'), Timestamp('2025-06-15 00:00:00'), Timestamp('2025-06-16 00:00:00'), Timestamp('2025-06-17 00:00:00'), Timestamp('2025-06-18 00:00:00'), Timestamp('2025-06-19 00:00:00'), Timestamp('2025-06-20 00:00:00'), Timestamp('2025-06-21 00:00:00')

In [12]:
reference_ticker = "NVDA"

reference_missing = set(
    missing_dates_by_ticker[reference_ticker]
)

for ticker, missing_dates in missing_dates_by_ticker.items():

    same_as_reference = (
        set(missing_dates) == reference_missing
    )

    print(
        ticker,
        "same missing dates as NVDA:",
        same_as_reference
    )

BRK-B same missing dates as NVDA: True
CVX same missing dates as NVDA: True
GE same missing dates as NVDA: True
MSFT same missing dates as NVDA: True
NVDA same missing dates as NVDA: True


In [13]:
missing_records = []
for ticker, missing_dates in missing_dates_by_ticker.items():
    for date in missing_dates:

        missing_records.append({
            "ticker": ticker,
            "date": date,
            "year": date.year})

missing_dates_df = pd.DataFrame(missing_records)
display(missing_dates_df.sort_values(["date", "ticker"]))

,ticker,date,year
0,BRK-B,2020-10-20,2020
19,CVX,2020-10-20,2020
38,GE,2020-10-20,2020
57,MSFT,2020-10-20,2020
76,NVDA,2020-10-20,2020
...,...,...,...
18,BRK-B,2025-07-01,2025
37,CVX,2025-07-01,2025
56,GE,2025-07-01,2025
75,MSFT,2025-07-01,2025


In [14]:
common_missing_dates = (
    missing_dates_df
    .groupby("date")["ticker"]
    .nunique()
    .reset_index(name="missing_ticker_count"))

common_missing_dates = common_missing_dates[common_missing_dates["missing_ticker_count"] == 5]

display(common_missing_dates)

,date,missing_ticker_count
0,2020-10-20,5
1,2023-03-23,5
2,2025-06-15,5
3,2025-06-16,5
4,2025-06-17,5
5,2025-06-18,5
6,2025-06-19,5
7,2025-06-20,5
8,2025-06-21,5
9,2025-06-22,5


In [15]:
unique_missing_dates = (
    missing_dates_df["date"]
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

print("Unique missing dates:", len(unique_missing_dates))
print(unique_missing_dates.tolist())

missing_count_by_date = (
    missing_dates_df
    .groupby("date")["ticker"]
    .nunique()
    .reset_index(name="missing_ticker_count")
)

display(missing_count_by_date)

Unique missing dates: 19
[Timestamp('2020-10-20 00:00:00'), Timestamp('2023-03-23 00:00:00'), Timestamp('2025-06-15 00:00:00'), Timestamp('2025-06-16 00:00:00'), Timestamp('2025-06-17 00:00:00'), Timestamp('2025-06-18 00:00:00'), Timestamp('2025-06-19 00:00:00'), Timestamp('2025-06-20 00:00:00'), Timestamp('2025-06-21 00:00:00'), Timestamp('2025-06-22 00:00:00'), Timestamp('2025-06-23 00:00:00'), Timestamp('2025-06-24 00:00:00'), Timestamp('2025-06-25 00:00:00'), Timestamp('2025-06-26 00:00:00'), Timestamp('2025-06-27 00:00:00'), Timestamp('2025-06-28 00:00:00'), Timestamp('2025-06-29 00:00:00'), Timestamp('2025-06-30 00:00:00'), Timestamp('2025-07-01 00:00:00')]


,date,missing_ticker_count
0,2020-10-20,5
1,2023-03-23,5
2,2025-06-15,5
3,2025-06-16,5
4,2025-06-17,5
5,2025-06-18,5
6,2025-06-19,5
7,2025-06-20,5
8,2025-06-21,5
9,2025-06-22,5


In [16]:
raw_output_file = (
    "/kaggle/working/"
    "gdelt_all_tickers_raw_2020_2025.csv")

gdelt_raw = gdelt_combined[
    [   "date",
        "ticker",
        "gdelt_company_tone"]].copy()
gdelt_raw.to_csv(raw_output_file,index=False)

print("Saved:", raw_output_file)
print("Raw shape:", gdelt_raw.shape)

Saved: /kaggle/working/gdelt_all_tickers_raw_2020_2025.csv
Raw shape: (10865, 3)


In [17]:
start_date = "2020-01-01"
end_date = "2025-12-31"

complete_date_range = pd.date_range(
    start=start_date,
    end=end_date,
    freq="D"
)

complete_tone_dfs = []

for ticker in gdelt_combined["ticker"].unique():

    ticker_df = (
        gdelt_combined[
            gdelt_combined["ticker"] == ticker
        ]
        .drop(columns=["year"], errors="ignore")
        .set_index("date")
        .sort_index()
    )

    ticker_df = ticker_df.reindex(
        complete_date_range
    )

    ticker_df.index.name = "date"

    ticker_df["ticker"] = ticker

    ticker_df = (
        ticker_df
        .reset_index()
    )

    complete_tone_dfs.append(ticker_df)

gdelt_daily = pd.concat(
    complete_tone_dfs,
    ignore_index=True
)

gdelt_daily = (
    gdelt_daily
    .sort_values(["ticker", "date"])
    .reset_index(drop=True)
)

print("Complete calendar shape:", gdelt_daily.shape)

display(gdelt_daily.head())

Complete calendar shape: (10960, 3)


,date,ticker,gdelt_company_tone
0,2020-01-01,BRK-B,0.4557
1,2020-01-02,BRK-B,-0.4681
2,2020-01-03,BRK-B,0.6690
3,2020-01-04,BRK-B,0.8500
4,2020-01-05,BRK-B,0.8177


In [18]:
print("Expected rows:", len(complete_date_range) * 5)
print("Actual rows:",len(gdelt_daily))

Expected rows: 10960
Actual rows: 10960


In [19]:
# Mark dates where GDELT did not return a tone value
gdelt_daily["gdelt_tone_missing"] = (gdelt_daily["gdelt_company_tone"].isna().astype(int))

print("Missing tone rows:",gdelt_daily["gdelt_tone_missing"].sum())

Missing tone rows: 95


In [20]:
gdelt_daily = (gdelt_daily.sort_values(["ticker", "date"]).reset_index(drop=True))

#Previous-day Tone
gdelt_daily["gdelt_tone_lag_1"] = (gdelt_daily.groupby("ticker")["gdelt_company_tone"].shift(1))

#Two-day Lag
gdelt_daily["gdelt_tone_lag_2"] = (gdelt_daily.groupby("ticker")["gdelt_company_tone"].shift(2))
#Three-day Lag
gdelt_daily["gdelt_tone_lag_3"] = (gdelt_daily.groupby("ticker")["gdelt_company_tone"].shift(3))

In [21]:
#previous-period rolling tone averages.
gdelt_daily["gdelt_tone_mean_3"] = (
    gdelt_daily
    .groupby("ticker")["gdelt_company_tone"]
    .transform(lambda x: (x.shift(1).rolling(window=3,min_periods=1).mean())))

gdelt_daily["gdelt_tone_mean_5"] = (
    gdelt_daily
    .groupby("ticker")["gdelt_company_tone"]
    .transform(lambda x: (x.shift(1).rolling(window=5,min_periods=1).mean())))

In [22]:
#Create tone volatility
gdelt_daily["gdelt_tone_std_5"] = (
    gdelt_daily
    .groupby("ticker")["gdelt_company_tone"]
    .transform(lambda x: (x.shift(1).rolling(window=5,min_periods=2).std())))

In [23]:
#Create leakage-safe tone change
gdelt_daily["gdelt_tone_change_1d"] = (
    gdelt_daily["gdelt_tone_lag_1"]
    - gdelt_daily["gdelt_tone_lag_2"])

In [24]:
#Create abnormal tone
gdelt_daily["gdelt_tone_mean_20"] = (
    gdelt_daily
    .groupby("ticker")["gdelt_company_tone"]
    .transform(
        lambda x: (
            x.shift(2)
            .rolling(
                window=20,
                min_periods=10
            )
            .mean()
        )
    )
)
gdelt_daily["gdelt_abnormal_tone_20"] = (
    gdelt_daily["gdelt_tone_lag_1"]
    - gdelt_daily["gdelt_tone_mean_20"]
)

In [25]:
tone_feature_columns = [
    "date",
    "ticker",
    "gdelt_company_tone",
    "gdelt_tone_missing",
    "gdelt_tone_lag_1",
    "gdelt_tone_mean_3",
    "gdelt_tone_mean_5",
    "gdelt_tone_std_5",
    "gdelt_tone_change_1d",
    "gdelt_abnormal_tone_20"
]

gdelt_processed = gdelt_daily[
    tone_feature_columns
].copy()

display(gdelt_processed.head(25))

,date,ticker,gdelt_company_tone,gdelt_tone_missing,gdelt_tone_lag_1,gdelt_tone_mean_3,gdelt_tone_mean_5,gdelt_tone_std_5,gdelt_tone_change_1d,gdelt_abnormal_tone_20
0,2020-01-01,BRK-B,0.4557,0,NaN,NaN,NaN,NaN,NaN,NaN
1,2020-01-02,BRK-B,-0.4681,0,0.4557,0.455700,0.455700,NaN,NaN,NaN
2,2020-01-03,BRK-B,0.6690,0,-0.4681,-0.006200,-0.006200,0.653225,-0.9238,NaN
3,2020-01-04,BRK-B,0.8500,0,0.6690,0.218867,0.218867,0.604414,1.1371,NaN
4,2020-01-05,BRK-B,0.8177,0,0.8500,0.350300,0.376650,0.585770,0.1810,NaN
5,2020-01-06,BRK-B,1.8944,0,0.8177,0.778900,0.464860,0.544289,-0.0323,NaN
6,2020-01-07,BRK-B,1.3205,0,1.8944,1.187367,0.752600,0.838828,1.0767,NaN
7,2020-01-08,BRK-B,0.5437,0,1.3205,1.344200,1.110320,0.501775,-0.5739,NaN
8,2020-01-09,BRK-B,0.2332,0,0.5437,1.252867,1.085260,0.531570,-0.7768,NaN
9,2020-01-10,BRK-B,-0.2012,0,0.2332,0.699133,0.961900,0.656666,-0.3105,NaN


In [26]:
#Check missing features and values
feature_missing_summary = (gdelt_processed.isna().sum().sort_values(ascending=False).reset_index())
feature_missing_summary.columns = ["column","missing_values"]
display(feature_missing_summary)

,column,missing_values
0,gdelt_abnormal_tone_20,200
1,gdelt_tone_change_1d,120
2,gdelt_tone_lag_1,100
3,gdelt_company_tone,95
4,gdelt_tone_std_5,85
5,gdelt_tone_mean_3,80
6,gdelt_tone_mean_5,70
7,date,0
8,ticker,0
9,gdelt_tone_missing,0


In [27]:
#Checking extreme tone values
tone_description = (
    gdelt_processed
    .groupby("ticker")[
        [
            "gdelt_company_tone",
            "gdelt_tone_lag_1",
            "gdelt_tone_mean_3",
            "gdelt_tone_mean_5",
            "gdelt_abnormal_tone_20"]].describe().T)
display(tone_description)

ticker                              BRK-B          CVX           GE  \
gdelt_company_tone     count  2173.000000  2173.000000  2173.000000   
                       mean      0.516666    -0.622693     0.592299   
                       std       0.887267     0.890236     0.903564   
                       min      -6.024900    -5.272400    -3.784600   
                       25%       0.019200    -1.097500     0.164100   
                       50%       0.596400    -0.534900     0.700000   
                       75%       1.053700    -0.045000     1.131200   
                       max       3.923600     2.299800     3.880500   
gdelt_tone_lag_1       count  2172.000000  2172.000000  2172.000000   
                       mean      0.516672    -0.622555     0.592031   
                       std       0.887472     0.890417     0.903685   
                       min      -6.024900    -5.272400    -3.784600   
                       25%       0.019175    -1.099450     0.163825   
                       50%       0.597250    -0.534350     0.699450   
                       75%       1.053825    -0.044975     1.130150   
                       max       3.923600     2.299800     3.880500   
gdelt_tone_mean_3      count  2176.000000  2176.000000  2176.000000   
                       mean      0.517364    -0.622635     0.590160   
                       std       0.653928     0.655747     0.690578   
                       min      -3.144600    -3.414700    -3.147650   
                       25%       0.149333    -1.001275     0.242975   
                       50%       0.593683    -0.557100     0.673150   
                       75%       0.943475    -0.169967     1.012375   
                       max       2.702800     1.173433     3.335300   
gdelt_tone_mean_5      count  2178.000000  2178.000000  2178.000000   
                       mean      0.518075    -0.623001     0.588639   
                       std       0.565977     0.571355     0.618394   
                       min      -2.470880    -2.797760    -3.147650   
                       25%       0.193030    -0.955340     0.270145   
                       50%       0.581340    -0.562760     0.650280   
                       75%       0.902205    -0.234380     0.976805   
                       max       2.702800     0.809740     3.047940   
gdelt_abnormal_tone_20 count  2152.000000  2152.000000  2152.000000   
                       mean     -0.001927     0.001690     0.005208   
                       std       0.863699     0.859011     0.863285   
                       min      -7.195760    -4.379084    -4.674815   
                       25%      -0.460459    -0.460238    -0.381812   
                       50%       0.054598     0.092882     0.098667   
                       75%       0.527856     0.540019     0.522894   
                       max       3.204050     3.200500     2.932479   

ticker                               MSFT         NVDA  
gdelt_company_tone     count  2173.000000  2173.000000  
                       mean      0.325812     0.593061  
                       std       0.626547     0.641914  
                       min      -3.761700    -3.257100  
                       25%      -0.007100     0.242800  
                       50%       0.389200     0.668000  
                       75%       0.714100     1.011800  
                       max       4.282900     3.182900  
gdelt_tone_lag_1       count  2172.000000  2172.000000  
                       mean      0.325991     0.593436  
                       std       0.626636     0.641824  
                       min      -3.761700    -3.257100  
                       25%      -0.006875     0.243175  
                       50%       0.389250     0.668100  
                       75%       0.714325     1.011975  
                       max       4.282900     3.182900  
gdelt_tone_mean_3      count  2176.000000  2176.000000  
                       mean      0.326211     0.594035  
     

In [28]:
display(gdelt_processed.nlargest(10,"gdelt_company_tone"))

,date,ticker,gdelt_company_tone,gdelt_tone_missing,gdelt_tone_lag_1,gdelt_tone_mean_3,gdelt_tone_mean_5,gdelt_tone_std_5,gdelt_tone_change_1d,gdelt_abnormal_tone_20
7631,2022-11-21,MSFT,4.2829,0,-0.8106,-0.017333,0.32294,0.706993,-1.0210,-1.344495
318,2020-11-14,BRK-B,3.9236,0,0.8389,1.449767,1.03654,0.718159,-1.2442,-0.148985
6158,2024-11-09,GE,3.8805,0,1.1253,2.190667,2.64186,1.033398,-0.9528,-0.563625
6153,2024-11-04,GE,3.6424,0,1.8134,2.173867,1.89170,0.999006,-1.6070,0.443985
6159,2024-11-10,GE,3.5895,0,3.8805,2.361300,2.68948,1.094698,2.7552,2.192420
937,2022-07-26,BRK-B,3.4677,0,-0.2547,0.498167,0.75722,0.664317,-0.7125,-0.487235
6151,2024-11-02,GE,3.4204,0,1.2878,1.408233,1.41894,0.881549,-0.8611,0.167405
6155,2024-11-06,GE,3.3686,0,2.9949,2.816900,2.63178,1.030917,-0.6475,1.534480
7085,2021-05-24,MSFT,3.3575,0,1.2536,0.394567,0.28000,0.583930,1.2963,1.436135
5760,2023-10-08,GE,3.2173,0,0.0274,0.856300,0.91312,0.537069,-1.2096,-0.904625


In [29]:
print("Final columns:")
print(gdelt_processed.columns.tolist())

print("\nTickers:")
print(gdelt_processed["ticker"].unique())

print("\nDate range:")
print(gdelt_processed["date"].min(),"to",gdelt_processed["date"].max())

print("\nTotal rows:", len(gdelt_processed))
print("Duplicate ticker-date rows:",gdelt_processed.duplicated(subset=["ticker", "date"]).sum())
print("Raw tone missing rows:",gdelt_processed["gdelt_company_tone"].isna().sum())

print("Missing indicator total:",gdelt_processed["gdelt_tone_missing"].sum())

Final columns:
['date', 'ticker', 'gdelt_company_tone', 'gdelt_tone_missing', 'gdelt_tone_lag_1', 'gdelt_tone_mean_3', 'gdelt_tone_mean_5', 'gdelt_tone_std_5', 'gdelt_tone_change_1d', 'gdelt_abnormal_tone_20']

Tickers:
['BRK-B' 'CVX' 'GE' 'MSFT' 'NVDA']

Date range:
2020-01-01 00:00:00 to 2025-12-31 00:00:00

Total rows: 10960
Duplicate ticker-date rows: 0
Raw tone missing rows: 95
Missing indicator total: 95


In [30]:
indicator_is_correct = (
    gdelt_processed["gdelt_tone_missing"]
    ==
    gdelt_processed["gdelt_company_tone"]
    .isna()
    .astype(int)
).all()

print("Missing indicator correctly matches raw tone:",indicator_is_correct)

Missing indicator correctly matches raw tone: True


In [31]:
processed_output_file = (
    "/kaggle/working/"
    "gdelt_all_tickers_processed_2020_2025.csv"
)

gdelt_processed.to_csv(
    processed_output_file,
    index=False
)

print("Saved:", processed_output_file)
print("Final shape:", gdelt_processed.shape)

Saved: /kaggle/working/gdelt_all_tickers_processed_2020_2025.csv
Final shape: (10960, 10)


**RAW-Combined-File**

* Actual GDELT observations only
* No artificially inserted dates
* No feature engineering


==================================

**Processed-combined file**

* Complete calendar
* Missing-tone indicator
* Lagged tone features
* Rolling tone features
* Abnormal tone


In [32]:
raw_df = pd.read_csv("/kaggle/working/gdelt_all_tickers_raw_2020_2025.csv")

print("Raw combined columns:")
raw_df.columns.tolist()

Raw combined columns:


['date', 'ticker', 'gdelt_company_tone']

In [33]:
processed_df = pd.read_csv("/kaggle/working/gdelt_all_tickers_processed_2020_2025.csv")

print("Processed combined columns:")
processed_df.columns.tolist()

Processed combined columns:


['date',
 'ticker',
 'gdelt_company_tone',
 'gdelt_tone_missing',
 'gdelt_tone_lag_1',
 'gdelt_tone_mean_3',
 'gdelt_tone_mean_5',
 'gdelt_tone_std_5',
 'gdelt_tone_change_1d',
 'gdelt_abnormal_tone_20']